# Tutorial: FalkorDB entity processing and retrieval

This notebook shows a minimal OGRE KG workflow on **FalkorDB**:

- ingest a small Wikipedia-based graph into FalkorDB
- create FalkorDB full-text indices for retrieval
- run a simple exact-match entity disambiguation pass
- retrieve graph context with FalkorDB-specific retrievers


## 1. Start FalkorDB locally with Docker

Run this in a terminal before executing the notebook:

```bash
docker run --rm -it --name ogre-falkordb-demo \
  -p 6379:6379 \
  -p 3000:3000 \
  -p 7687:7687 \
  -e FALKORDB_ARGS="BOLT_PORT 7687" \
  falkordb/falkordb:latest
```

This image includes the FalkorDB Browser at `http://localhost:3000`.

Notes:

- This command is intended for **local development/tutorial use**.
- The notebook below connects through Bolt at `bolt://localhost:7687`.
- If port `7687` is already used by Neo4j on your machine, stop Neo4j first or change both the Docker port mapping and the notebook connection URL.


## 2. Environment variables

This notebook uses OpenRouter for both the LLM and embeddings. Make sure `OPENROUTER_API_KEY` is available in your environment or `.env` file.

In [1]:

import autoroot  # noqa
import nest_asyncio
from dotenv import load_dotenv
from llama_index.core import PropertyGraphIndex, StorageContext
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.core.prompts import PromptTemplate
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.graph_stores.falkordb import FalkorDBPropertyGraphStore
from llama_index.llms.openai import OpenAI
from llama_index.readers.wikipedia import WikipediaReader
from rich import print as rprint

from ogre_kg.kg_processors import make_graph_text_index_builder
from ogre_kg.kg_processors.entity_processing import (
    CanonicalEntitySelectionStrategy,
    EntityDisambiguationProcessor,
    ExactMatchEntitySimilarityFinder,
    FalkorDBEntityMerger,
    FuzzyEntitySimilarityFinder,
)
from ogre_kg.kg_processors.extractors import OntoDynamicLLMPathExtractor
from ogre_kg.kg_processors.retrievers import (
    FalkorDBChunkKeywordRetriever,
    FalkorDBKeywordContextRetriever,
)

In [11]:
load_dotenv(override=True)
nest_asyncio.apply()

## 3. LLM and embedding setup

In [2]:
llm = OpenAI(model="gpt-5-mini")
embedding = OpenAIEmbedding(
    model_name="text-embedding-3-small",)

## 4. Fetch a tiny Wikipedia dataset

Two short pages are enough here. The goal is not benchmark quality, only to have a small graph that is easy to inspect in the FalkorDB Browser.

In [3]:
reader = WikipediaReader()
documents = reader.load_data(pages=["deep learning", "artificial intelligence"])
splitter = SemanticSplitterNodeParser(embed_model=embedding)
chunks = splitter.get_nodes_from_documents(documents)

len(documents), len(chunks)

(2, 51)

In [4]:
rprint(chunks[0].get_content()[:1500])

In machine learning, deep learning (DL) focuses on utilizing multilayered neural networks to perform tasks such as 
classification, regression, and representation learning. The field takes inspiration from biological neuroscience 
and revolves around stacking artificial neurons into layers and "training" them to process data. The adjective 
"deep" refers to the use of multiple layers (ranging from three to several hundred or thousands) in the network. 
Methods used can be supervised, semi-supervised or unsupervised.
Some common deep learning network architectures include fully connected networks, deep belief networks, recurrent 
neural networks, convolutional neural networks, generative adversarial networks, transformers, and neural radiance 
fields. These architectures have been applied to fields including computer vision, speech recognition, natural 
language processing, machine translation, bioinformatics, drug design, medical image analysis, climate science, 
material inspection and board game programs, where they have produced results comparable to and in some cases 
surpassing human expert performance.
Early forms of neural networks were inspired by information processing and distributed communication nodes in 
biological systems, particularly the human brain. However, current neural networks do not intend to model the brain
function of organisms, and are generally seen as low-quality models for that purpose.


== Overview ==
Most modern deep learning models are based on multi-la

In [14]:
chunks_sample = chunks[:5]

## 5. Connect to FalkorDB

The default FalkorDB graph name used below is `falkor`.

In [9]:
graph_store = FalkorDBPropertyGraphStore(
    url="falkor://localhost:6379",
    database="falkor",
)

graph_store.structured_query("MATCH (n) RETURN COUNT(n) AS node_count")

[{'node_count': 0}]

## 6. Build a fresh OGRE KG index in FalkorDB

Set `recreate_index = True` to clear the graph and re-ingest from scratch.

In [95]:
recreate_index = True

extractor = OntoDynamicLLMPathExtractor(
    llm=llm,
    max_triplets_per_chunk=5,
    num_workers=5,
)

if recreate_index:
    graph_store.structured_query("MATCH (n) DETACH DELETE n")
    storage_context = StorageContext.from_defaults(property_graph_store=graph_store)
    index = PropertyGraphIndex(
        nodes=chunks_sample,
        embed_model=embedding,
        kg_extractors=[extractor],
        storage_context=storage_context,
        show_progress=True,
    )
else:
    index = PropertyGraphIndex.from_existing(
        property_graph_store=graph_store,
        embed_model=embedding,
        kg_extractors=[extractor],
        show_progress=False,
    )

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings: 100%|██████████| 1/1 [00:01<00:00,  1.22s/it]


## 7. Create FalkorDB full-text indices

These indices are used by the FalkorDB keyword and chunk retrievers.

In [ ]:
builder = make_graph_text_index_builder(graph_store)
builder.create_indices()

## 8. Exact-match disambiguation in FalkorDB

The exact-match finder normalizes names by trimming whitespace, collapsing repeated whitespace, and case-folding. The FalkorDB merger then keeps the canonical node selected by the configured strategy. Here the default strategy is `MOST_CONNECTED`.

In [97]:
exact_finder = ExactMatchEntitySimilarityFinder(graph_store)
exact_groups = exact_finder.find_similar_entities()
exact_groups[:10]

[]

In [98]:
falkordb_merger = FalkorDBEntityMerger(
    graph_store,
    canonical_selection_strategy=CanonicalEntitySelectionStrategy.MOST_CONNECTED,
    preview_changes=False,
)

disambiguator = EntityDisambiguationProcessor(exact_finder, falkordb_merger)
merge_results = disambiguator.process()
merge_results

[]

## 9. Optional: fuzzy similarity inspection

You mentioned threshold-driven strategies. The fuzzy finder is backend-agnostic and can also be used on FalkorDB because it reads entity nodes through the graph-store `get()` interface. This cell only inspects candidate fuzzy groups; it does not modify the graph.

In [100]:
fuzzy_finder = FuzzyEntitySimilarityFinder(graph_store, similarity_threshold=85.0)
fuzzy_groups = fuzzy_finder.find_similar_entities()
fuzzy_groups[:10]

[{'cycles in connectivity', 'no cycles in connectivity'}]

In [101]:
falkordb_merger = FalkorDBEntityMerger(
    graph_store,
    canonical_selection_strategy=CanonicalEntitySelectionStrategy.MOST_CONNECTED,
    preview_changes=False,
)

disambiguator = EntityDisambiguationProcessor(fuzzy_finder, falkordb_merger)
merge_results = disambiguator.process()
merge_results

[{'canonical_name': 'cycles in connectivity',
  'canonical_node_id': 28,
  'merged_entities': ['cycles in connectivity', 'no cycles in connectivity'],
  'deleted_node_ids': [27],
  'relationships_created': 2,
  'merged_labels': ['PROPERTY', '__Entity__'],
  'merged_properties': {'id': 'cycles in connectivity',
   'triplet_source_id': '06455a98-748c-412e-b52f-a14d9a54e9b3',
   'name': 'cycles in connectivity'}}]

## 10. FalkorDB retrievers

Two FalkorDB-specific retrievers are available:

- `FalkorDBKeywordContextRetriever` seeds traversal from full-text entity-name hits
- `FalkorDBChunkKeywordRetriever` seeds traversal from full-text chunk-text hits


In [102]:
keyword_prompt = PromptTemplate(
    "Generate up to {max_keywords} graph lookup keywords for the user question.\n"
    "Return only concise entity names or graph terms.\n"
    "QUESTION: {question}"
)
chunk_terms_prompt = PromptTemplate(
    "Generate up to {max_terms} concise chunk fulltext search phrases for the user question.\n"
    "Prefer phrases that are likely to appear verbatim in the source text.\n"
    "QUESTION: {question}"
)

keyword_retriever = FalkorDBKeywordContextRetriever(
    graph_store,
    llm=llm,
    topk_search=3,
    keyword_prompt=keyword_prompt,
)
chunk_retriever = FalkorDBChunkKeywordRetriever(
    graph_store,
    llm=llm,
    topk_search=3,
    max_chunk_terms=2,
    restrict_to_seed_chunks=True,
    keyword_prompt=keyword_prompt,
    chunk_terms_prompt=chunk_terms_prompt,
)

In [103]:
question = 'What is the vanishing gradient problem in deep learning?'
kw_nodes = keyword_retriever.retrieve(question)
chunk_nodes = chunk_retriever.retrieve(question)

len(kw_nodes), len(chunk_nodes)

(2, 1)

In [104]:
rprint(kw_nodes[0].get_content())

Here are some facts extracted from the provided text:

Paul Werbos -> APPLIED_TO -> backpropagation
David E. Rumelhart et al. -> POPULARIZED -> backpropagation
Time Delay Neural Network -> USES -> backpropagation

republished it in 1971. Paul Werbos applied backpropagation to neural networks in 1982 (his 1974 PhD thesis, 
reprinted in a 1994 book, did not yet describe the algorithm). In 1986, David E. Rumelhart et al. popularised 
backpropagation but did not cite the original work.


=== 1980s-2000s ===
The time delay neural network (TDNN) was introduced in 1987 by Alex Waibel to apply CNN to phoneme recognition. It 
used convolutions, weight sharing, and backpropagation.  In 1988, Wei Zhang applied a backpropagation-trained CNN 
to alphabet recognition. 
In 1989, Yann LeCun et al. created a CNN called LeNet for recognizing handwritten ZIP codes on mail. Training 
required 3 days. In 1990, Wei Zhang implemented a CNN on optical computing hardware. In 1991, a CNN was applied to 
medical image object segmentation and breast cancer detection in mammograms. LeNet-5 (1998), a 7-level CNN by Yann 
LeCun et al., that classifies digits, was applied by several banks to recognize hand-written numbers on checks  
digitized in 32x32 pixel images.
Recurrent neural networks (RNN) were further developed in the 1980s.

## 11. Ask the graph-backed query engine

In [105]:
query_engine = index.as_query_engine(
    sub_retrievers=[chunk_retriever, keyword_retriever],
    llm=llm,
)
response = query_engine.query(question)
rprint(response.response)

The vanishing gradient problem is the difficulty of training very deep networks (including recurrent networks 
unrolled in time) using backpropagation when the required credit assignment spans many layers or time steps. 
Gradients propagated backward become ineffective for assigning credit across long “credit‑assignment paths,” so the
network cannot learn long-range dependencies. It was identified and analyzed by Sepp Hochreiter (1991). Proposed 
remedies include recurrent residual connections and architectures such as LSTM (developed afterward), which can 
handle very long credit‑assignment paths and retain information across thousands of time steps.

In [87]:
for node in response.source_nodes:
    rprint(node.get_content()[:500])
    rprint("---")

Here are some facts extracted from the provided text:

Sepp Hochreiter -> IDENTIFIED -> vanishing gradient problem
Jürgen Schmidhuber -> PROPOSED -> hierarchy of RNNs
Elman network -> APPLIED_TO -> cognitive psychology
Jordan network -> APPLIED_TO -> cognitive psychology
unrolled recurrent network -> RESEMBLES -> deep feedforward layer

Recurrence is used for sequence processing, and when a recurrent network is unrolled, it mathematically resembles a
deep feedforward layer. Consequently, they ha

---

Here are some facts extracted from the provided text:

Sepp Hochreiter -> IDENTIFIED -> vanishing gradient problem

Recurrence is used for sequence processing, and when a recurrent network is unrolled, it mathematically resembles a
deep feedforward layer. Consequently, they have similar properties and issues, and their developments had mutual 
influences. In RNN, two early influential works were the Jordan network (1986) and the Elman network (1990), which 
applied RNN to study problems in cogniti

---

Here are some facts extracted from the provided text:

Time delay neural network -> USES -> Backpropagation

republished it in 1971. Paul Werbos applied backpropagation to neural networks in 1982 (his 1974 PhD thesis, 
reprinted in a 1994 book, did not yet describe the algorithm). In 1986, David E. Rumelhart et al. popularised 
backpropagation but did not cite the original work.


=== 1980s-2000s ===
The time delay neural network (TDNN) was introduced in 1987 by Alex Waibel to apply CNN to phoneme

---

Here are some facts extracted from the provided text:

Backpropagation -> FIRST_PUBLISHED_IN -> Seppo Linnainmaa master thesis

It features inference, as well as the optimization concepts of training and testing, related to fitting and 
generalization, respectively. More specifically, the probabilistic interpretation considers the activation 
nonlinearity as a cumulative distribution function. The probabilistic interpretation led to the introduction of 
dropout as regularizer in neural networks. Th

---

## 12. Inspect the graph in the FalkorDB Browser

Open `http://localhost:3000` and try queries such as:

```cypher
MATCH (n) RETURN labels(n), count(*)
```

```cypher
MATCH (n:__Entity__) WHERE toLower(n.name) CONTAINS 'deep learning' RETURN n
```

```cypher
MATCH (c:Chunk)-[r]->(e:__Entity__)
WHERE toLower(c.text) CONTAINS 'vanishing gradient'
RETURN c, r, e
```

That should give you a quick visual check that the indexing, merge, and retrieval steps are behaving as expected.